In [40]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os
import joblib  # en üstte importların arasına ekleyebilirsi

In [41]:
processed_path = "../data/processed/weather_processed.csv"
df = pd.read_csv(processed_path)

print("--- Cleaned Data Loaded for Feature Engineering ---")
print(f"Current Shape: {df.shape}")

--- Cleaned Data Loaded for Feature Engineering ---
Current Shape: (140787, 23)


In [42]:
# Convert Date to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Extract Month index (1 to 12)
df['Month'] = df['Date'].dt.month

# Apply Cyclical Encoding using Sine and Cosine
df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12.0)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12.0)

# Drop original Date and temporary Month columns
df = df.drop(['Date', 'Month'], axis=1)

print("--- Cyclical Temporal Encoding Complete ---")
print(df[['Month_Sin', 'Month_Cos']].head(3))

--- Cyclical Temporal Encoding Complete ---
      Month_Sin  Month_Cos
0 -2.449294e-16        1.0
1 -2.449294e-16        1.0
2 -2.449294e-16        1.0


In [43]:
# --- Yeni Meteorolojik Göstergeler ve Etkileşim Özellikleri ---

# 1. Temperature Range: Gün içindeki sıcaklık dalgalanması
df['Temp_Range'] = df['MaxTemp'] - df['MinTemp']

# 2. Humidity-Temperature Interaction: Nem ve Sıcaklık etkileşimi
df['Humidity_Temp_Interaction'] = df['Humidity3pm'] * df['MaxTemp']

# 3. Pressure Drop: Sabah ve öğleden sonra basınç farkı
df['Pressure_Drop'] = df['Pressure9am'] - df['Pressure3pm']

print("--- New Meteorological Indicators Created ---")
print(df[['Temp_Range', 'Humidity_Temp_Interaction', 'Pressure_Drop']].head(3))

--- New Meteorological Indicators Created ---
   Temp_Range  Humidity_Temp_Interaction  Pressure_Drop
0         9.5                      503.8            0.6
1        17.7                      627.5            2.8
2        12.8                      771.0           -1.1


In [44]:
# Select categorical columns to encode
categorical_cols = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

print("--- One-Hot Encoding Complete ---")
print(f"New Shape after encoding: {df_encoded.shape}")

--- One-Hot Encoding Complete ---
New Shape after encoding: (140787, 117)


In [45]:
import joblib

columns_to_scale = df_encoded.select_dtypes(include=['float64', 'int64']).columns
columns_to_scale = columns_to_scale.drop(['RainTomorrow', 'RainToday'])

scaler = StandardScaler()
df_encoded[columns_to_scale] = scaler.fit_transform(df_encoded[columns_to_scale])

joblib.dump(scaler, "../models/scaler.pkl")
print("--- Feature Scaling Complete ---")

--- Feature Scaling Complete ---


In [46]:
# Define final output path
final_output_path = "../data/processed/weather_final.csv"

# Save the fully processed feature matrix to disk
df_encoded.to_csv(final_output_path, index=False)

print("--- Feature Engineering Process Complete! ---")
print(f"Final fully-engineered dataset saved to: {final_output_path}")
print(f"Final Shape for Machine Learning: {df_encoded.shape}")

--- Feature Engineering Process Complete! ---
Final fully-engineered dataset saved to: ../data/processed/weather_final.csv
Final Shape for Machine Learning: (140787, 117)


In [47]:
print("--- FEATURE ENGINEERING SANITY CHECK ---")
print(f"1. Final Dataset Shape: {df_encoded.shape}")
print(f"2. Any Remaining Missing Values? {df_encoded.isnull().sum().sum()}")

# Check if object/string types are completely gone
non_numeric_cols = df_encoded.select_dtypes(include=['object']).columns.tolist()
print(f"3. Remaining String/Object Columns: {non_numeric_cols}")

# Check if scaling worked (Mean should be close to 0, Standard Deviation close to 1)
print("\n4. Scaling Verification (Sample - MaxTemp):")
print(f"   MaxTemp Mean: {df_encoded['MaxTemp'].mean():.4f} (Should be near 0)")
print(f"   MaxTemp Std Dev: {df_encoded['MaxTemp'].std():.4f} (Should be near 1)")

# Check new features are present
new_features = ['Temp_Range', 'Humidity_Temp_Interaction', 'Pressure_Drop']
missing_new = [f for f in new_features if f not in df_encoded.columns]
print(f"\n6. New Features Check:")
print(f"   Missing new features: {missing_new if missing_new else 'None - All present!'}")

print("\n5. Target Verification:")
print(f"   RainTomorrow unique values: {df_encoded['RainTomorrow'].unique().tolist()} (Should be [0, 1])")

print("\n--- First 2 Rows of Fully Engineered Data ---")
df_encoded.head(2)

--- FEATURE ENGINEERING SANITY CHECK ---
1. Final Dataset Shape: (140787, 117)
2. Any Remaining Missing Values? 0
3. Remaining String/Object Columns: []

4. Scaling Verification (Sample - MaxTemp):
   MaxTemp Mean: -0.0000 (Should be near 0)
   MaxTemp Std Dev: 1.0000 (Should be near 1)

6. New Features Check:
   Missing new features: None - All present!

5. Target Verification:
   RainTomorrow unique values: [0, 1] (Should be [0, 1])

--- First 2 Rows of Fully Engineered Data ---


,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,...,WindDir3pm_NNW,WindDir3pm_NW,WindDir3pm_S,WindDir3pm_SE,WindDir3pm_SSE,WindDir3pm_SSW,WindDir3pm_SW,WindDir3pm_W,WindDir3pm_WNW,WindDir3pm_WSW
0,0.189400,-0.047135,-0.252746,-0.136834,0.15985,0.359557,0.714030,0.650942,0.110364,-1.429805,...,-0.240815,-0.256077,-0.270242,-0.303285,-0.263029,-0.245972,-0.264354,-0.276589,3.928567,-0.265828
1,-0.748557,0.262305,-0.355049,-0.136834,0.15985,0.359557,-1.153499,0.415530,-1.310139,-1.284474,...,-0.240815,-0.256077,-0.270242,-0.303285,-0.263029,-0.245972,-0.264354,-0.276589,-0.254546,3.761837


In [48]:
import joblib, numpy as np
s = joblib.load("../models/scaler.pkl")
print("Scaler mean_ (ilk 5):", s.mean_[:5])
print("Scaler scale_ (ilk 5):", s.scale_[:5])

Scaler mean_ (ilk 5): [12.18843217 23.23511191  2.0823478   4.67892916  8.05163119]
Scaler scale_ (ilk 5): [6.39688344 7.10962907 5.86496061 2.0384429  2.80493113]
